# Build AI Conference Dataset

Fetch structured conference metadata from CCF Deadlines, filter to top AI/ML and adjacent venues for 2026 and 2027, enrich each row with coordinates and an importance score, then persist `../data/conferences.csv` for the globe visualization.

The pipeline uses structured public data first and a deterministic local geocode cache for known conference venues. A Nominatim fallback is included for new venues, with results cached locally in `../data/geocode_cache.json`.

In [1]:
from __future__ import annotations

import json
from io import BytesIO
import re
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
import yaml
from PIL import Image
from dateutil import parser as date_parser

CURRENT_YEAR = 2026
YEARS = {CURRENT_YEAR, CURRENT_YEAR + 1}
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
IMAGE_DIR = DATA_DIR / "images"
DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

CCF_API_ROOT = "https://api.github.com/repos/ccfddl/ccf-deadlines/contents/conference"
SOURCE_FOLDERS = ["AI", "DB"]

# Curated top academic AI/ML and adjacent venues. This keeps the globe readable.
TOP_TITLES = {
    "AAAI", "AAMAS", "ACL", "AISTATS", "BMVC", "CIKM", "COLM",
    "COLT", "CoNLL", "CoRL", "CVPR", "EACL", "ECCV", "EMNLP",
    "ICAPS", "ICDAR", "ICLR", "ICML", "ICRA", "ICDE", "ICDM",
    "IJCAI", "IROS", "KR", "NeurIPS", "SIGIR", "SIGKDD", "UAI",
    "WACV", "WSDM", "RSS", "RECSYS", "ECIR", "PAKDD", "SDM"
}

IMPORTANCE_OVERRIDES = {
    "NeurIPS": 10.0,
    "ICML": 9.6,
    "ICLR": 9.4,
    "CVPR": 9.2,
    "ACL": 8.8,
    "EMNLP": 8.4,
    "AAAI": 8.2,
    "IJCAI": 8.0,
    "ECCV": 8.0,
    "SIGKDD": 8.0,
    "ICRA": 7.8,
    "ICDE": 7.6,
    "SIGIR": 7.5,
    "AISTATS": 7.4,
    "COLT": 7.3,
    "UAI": 7.1,
    "CoRL": 7.0,
    "IROS": 6.9,
    "WSDM": 6.8,
    "CIKM": 6.7,
    "COLM": 6.7,
    "WACV": 6.5,
    "EACL": 6.4,
    "BMVC": 6.2,
    "AAMAS": 6.1,
    "RSS": 7.0,
}

RANK_IMPORTANCE = {"A*": 7.8, "A": 6.8, "B": 5.3, "C": 4.0, "N": 3.0}

KNOWN_COORDS = {
    "singapore": (1.3521, 103.8198, "Singapore", "Singapore"),
    "montréal": (45.5019, -73.5674, "Montréal", "Canada"),
    "montreal": (45.5019, -73.5674, "Montréal", "Canada"),
    "paphos": (34.7720, 32.4297, "Paphos", "Cyprus"),
    "san diego": (32.7157, -117.1611, "San Diego", "United States"),
    "morocco": (31.7917, -7.0926, "Morocco", "Morocco"),
    "rabat": (34.0209, -6.8416, "Rabat", "Morocco"),
    "toronto": (43.6532, -79.3832, "Toronto", "Canada"),
    "lancaster": (54.0466, -2.8007, "Lancaster", "United Kingdom"),
    "maastricht": (50.8514, 5.6910, "Maastricht", "Netherlands"),
    "san francisco": (37.7749, -122.4194, "San Francisco", "United States"),
    "tübingen": (48.5216, 9.0576, "Tübingen", "Germany"),
    "tubingen": (48.5216, 9.0576, "Tübingen", "Germany"),
    "denver": (39.7392, -104.9903, "Denver", "United States"),
    "malmö": (55.6050, 13.0038, "Malmö", "Sweden"),
    "malmo": (55.6050, 13.0038, "Malmö", "Sweden"),
    "budapest": (47.4979, 19.0402, "Budapest", "Hungary"),
    "bruges": (51.2093, 3.2247, "Bruges", "Belgium"),
    "luxembourg": (49.6116, 6.1319, "Luxembourg", "Luxembourg"),
    "kyoto": (35.0116, 135.7681, "Kyoto", "Japan"),
    "san josé": (9.9281, -84.0907, "San José", "Costa Rica"),
    "san jose": (9.9281, -84.0907, "San José", "Costa Rica"),
    "padua": (45.4064, 11.8768, "Padua", "Italy"),
    "dublin": (53.3498, -6.2603, "Dublin", "Ireland"),
    "bremen": (53.0793, 8.8017, "Bremen", "Germany"),
    "vienna": (48.2082, 16.3738, "Vienna", "Austria"),
    "rio de janeiro": (-22.9068, -43.1729, "Rio de Janeiro", "Brazil"),
    "brazil": (-22.9068, -43.1729, "Rio de Janeiro", "Brazil"),
    "seoul": (37.5665, 126.9780, "Seoul", "South Korea"),
    "melbourne": (-37.8136, 144.9631, "Melbourne", "Australia"),
    "lyon": (45.7640, 4.8357, "Lyon", "France"),
    "boca raton": (26.3683, -80.1289, "Boca Raton", "United States"),
    "rome": (41.9028, 12.4964, "Rome", "Italy"),
    "hengqin": (22.1194, 113.5439, "Hengqin", "China"),
    "pittsburgh": (40.4406, -79.9959, "Pittsburgh", "United States"),
    "lisbon": (38.7223, -9.1393, "Lisbon", "Portugal"),
    "beijing": (39.9042, 116.4074, "Beijing", "China"),
    "sydney": (-33.8688, 151.2093, "Sydney", "Australia"),
    "macau": (22.1987, 113.5439, "Macau", "China"),
    "trento": (46.0748, 11.1217, "Trento", "Italy"),
    "guangzhou": (23.1291, 113.2644, "Guangzhou", "China"),
    "vilnius": (54.6872, 25.2797, "Vilnius", "Lithuania"),
    "amsterdam": (52.3676, 4.9041, "Amsterdam", "Netherlands"),
    "tucson": (32.2226, -110.9747, "Tucson", "United States"),
    "boise": (43.6150, -116.2023, "Boise", "United States"),
    "jeju": (33.4996, 126.5312, "Jeju", "South Korea"),
    "copenhagen": (55.6761, 12.5683, "Copenhagen", "Denmark"),
}

print(f"Writing output under {DATA_DIR}")

Writing output under /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/AIConferences/data


In [2]:
def fetch_yaml_files(folder: str) -> list[dict]:
    response = requests.get(f"{CCF_API_ROOT}/{folder}?ref=main", timeout=30)
    response.raise_for_status()
    files = response.json()
    rows = []
    for file_info in files:
        if not file_info["name"].endswith((".yml", ".yaml")):
            continue
        raw = requests.get(file_info["download_url"], timeout=30)
        raw.raise_for_status()
        try:
            docs = yaml.safe_load(raw.text) or []
        except yaml.YAMLError as exc:
            print(f"Skipping {file_info['path']}: {exc}")
            continue
        for doc in docs:
            doc["_source_path"] = file_info["path"]
            rows.append(doc)
    return rows


def first_timeline(conference: dict) -> dict:
    timeline = conference.get("timeline") or []
    return timeline[-1] if timeline else {}


def clean_date(value) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    return "TBD" if text.upper() == "TBD" else text


def date_only(value: str) -> str:
    value = clean_date(value)
    if not value or value == "TBD":
        return value
    return value.split()[0]


MONTH_RE = re.compile(
    r"\b(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
    r"Aug(?:ust)?|Sep(?:t|tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\b",
    re.IGNORECASE,
)


def iso_date(dt) -> str:
    return dt.date().isoformat()


def parse_event_range(date_text: str, fallback_year: int) -> tuple[str, str]:
    text = clean_date(date_text)
    if not text or text == "TBD":
        return "", ""

    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\bSept\b", "Sep", text)
    year_match = re.search(r"(20\d{2})", text)
    year = int(year_match.group(1)) if year_match else int(fallback_year)
    body = re.sub(r",?\s*20\d{2}.*$", "", text).strip(" .,")

    try:
        if "-" not in body:
            dt = date_parser.parse(f"{body} {year}", fuzzy=True, default=date_parser.parse(f"Jan 1 {year}"))
            return iso_date(dt), iso_date(dt)

        left, right = [part.strip(" .,()") for part in body.split("-", 1)]
        start_dt = date_parser.parse(f"{left} {year}", fuzzy=True, default=date_parser.parse(f"Jan 1 {year}"))
        if MONTH_RE.search(right):
            end_dt = date_parser.parse(f"{right} {year}", fuzzy=True, default=date_parser.parse(f"Jan 1 {year}"))
        else:
            month_name = start_dt.strftime("%B")
            end_dt = date_parser.parse(f"{month_name} {right} {year}", fuzzy=True, default=date_parser.parse(f"Jan 1 {year}"))
            if end_dt < start_dt:
                end_dt = date_parser.parse(f"{month_name} {right} {year + 1}", fuzzy=True)
        return iso_date(start_dt), iso_date(end_dt)
    except (ValueError, TypeError, OverflowError):
        return "", ""


def rank_label(rank: dict | None) -> str:
    rank = rank or {}
    for key in ["ccf", "core", "thcpl"]:
        if rank.get(key):
            return str(rank[key])
    return ""


def importance_for(title: str, rank: str) -> float:
    if title in IMPORTANCE_OVERRIDES:
        return IMPORTANCE_OVERRIDES[title]
    return RANK_IMPORTANCE.get(rank, 4.5)


def infer_subfield(title: str, default: str) -> str:
    mapping = {
        "CVPR": "CV", "ECCV": "CV", "WACV": "CV", "BMVC": "CV", "ICDAR": "CV",
        "ACL": "NLP", "EMNLP": "NLP", "EACL": "NLP", "CoNLL": "NLP", "COLM": "NLP",
        "ICRA": "RO", "IROS": "RO", "RSS": "RO", "CoRL": "RO",
        "SIGKDD": "DM", "SIGIR": "IR", "WSDM": "DM", "CIKM": "DM", "ICDE": "DB",
    }
    return mapping.get(title, default or "AI/ML")


def normalize_title(title: str) -> str:
    return "NeurIPS" if title == "NIPS" else title


def parse_records() -> pd.DataFrame:
    records = []
    for folder in SOURCE_FOLDERS:
        for entry in fetch_yaml_files(folder):
            title = normalize_title(entry.get("title", ""))
            if title not in TOP_TITLES:
                continue
            rank = rank_label(entry.get("rank"))
            for conf in entry.get("confs", []) or []:
                if conf.get("year") not in YEARS:
                    continue
                timeline = first_timeline(conf)
                deadline = clean_date(timeline.get("deadline"))
                abstract_deadline = clean_date(timeline.get("abstract_deadline"))
                event_start = date_only(conf.get("start", ""))
                event_end = date_only(conf.get("end", ""))
                if not event_start:
                    event_start, event_end = parse_event_range(conf.get("date", ""), conf.get("year"))
                records.append({
                    "id": conf.get("id") or f"{title.lower()}{conf.get('year')}",
                    "title": title,
                    "full_name": entry.get("description", title),
                    "year": conf.get("year"),
                    "subfield": infer_subfield(title, entry.get("sub", "")),
                    "rank": rank,
                    "importance": importance_for(title, rank),
                    "link": conf.get("link", ""),
                    "source": f"CCF Deadlines:{entry.get('_source_path')}",
                    "deadline": deadline,
                    "abstract_deadline": abstract_deadline,
                    "deadline_timezone": conf.get("timezone", ""),
                    "event_start": event_start,
                    "event_end": event_end,
                    "date_text": conf.get("date", ""),
                    "place": conf.get("place", ""),
                    "deadline_status": "TBD" if deadline == "TBD" else ("known" if deadline else "missing"),
                    "event_status": "TBD" if event_start == "TBD" else ("known" if event_start else "missing"),
                    "notes": timeline.get("comment", ""),
                })
    df = pd.DataFrame(records)
    return df.sort_values(["year", "importance", "title"], ascending=[True, False, True]).reset_index(drop=True)

df = parse_records()
df[["title", "year", "date_text", "place", "importance"]].head(12)

,title,year,date_text,place,importance
0,NeurIPS,2026,"December 6, 2026","Sydney, Australia",10.0
1,ICML,2026,"July 6-12, 2026","Seoul, Korea",9.6
2,ICLR,2026,"May 01-05, 2026",Brazil,9.4
3,CVPR,2026,"June 3-7, 2026","Denver, Colorado, United States",9.2
4,ACL,2026,"July 2 - 7, 2026","San Diego, California, United States",8.8
5,EMNLP,2026,"October 24 - 29, 2026","Budapest, Hungary",8.4
6,AAAI,2026,"January 20 - 27, 2026",Singapore EXPO,8.2
7,ECCV,2026,"September 8 - 13, 2026","Malmö, Sweden",8.0
8,IJCAI,2026,"August 15-21, 2026","Bremen, Germany",8.0
9,SIGKDD,2026,"August 9-13, 2026","Jeju, Korea",8.0


In [3]:
def load_geocode_cache() -> dict:
    cache_path = DATA_DIR / "geocode_cache.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    return {}


def save_geocode_cache(cache: dict) -> None:
    cache_path = DATA_DIR / "geocode_cache.json"
    cache_path.write_text(json.dumps(cache, indent=2, sort_keys=True))


def normalize_place_key(place: str) -> str:
    text = (place or "").lower()
    text = text.replace("united states", "usa")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def known_geocode(place: str):
    key = normalize_place_key(place)
    if not key or "virtual" in key or "online" == key:
        return None
    for needle, value in KNOWN_COORDS.items():
        if needle in key:
            lat, lon, city, country = value
            return {"latitude": lat, "longitude": lon, "city": city, "country": country}
    return None


def nominatim_geocode(place: str, cache: dict):
    key = normalize_place_key(place)
    if not key:
        return None
    if key in cache:
        return cache[key]

    response = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": place, "format": "jsonv2", "limit": 1, "addressdetails": 1},
        headers={"User-Agent": "AIConferenceGlobe/1.0 contact: local-notebook"},
        timeout=20,
    )
    if not response.ok or not response.json():
        cache[key] = None
        return None

    item = response.json()[0]
    address = item.get("address", {})
    result = {
        "latitude": float(item["lat"]),
        "longitude": float(item["lon"]),
        "city": address.get("city") or address.get("town") or address.get("state") or "",
        "country": address.get("country", ""),
    }
    cache[key] = result
    time.sleep(1.0)
    return result


def geocode_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cache = load_geocode_cache()
    enriched = []
    for row in df.to_dict(orient="records"):
        geo = known_geocode(row["place"]) or nominatim_geocode(row["place"], cache)
        row.update(geo or {"latitude": "", "longitude": "", "city": "", "country": ""})
        enriched.append(row)
    save_geocode_cache(cache)
    return pd.DataFrame(enriched)

df = geocode_dataframe(df)
df[["title", "year", "place", "latitude", "longitude"]].head(12)

,title,year,place,latitude,longitude
0,NeurIPS,2026,"Sydney, Australia",-33.8688,151.2093
1,ICML,2026,"Seoul, Korea",37.5665,126.9780
2,ICLR,2026,Brazil,-22.9068,-43.1729
3,CVPR,2026,"Denver, Colorado, United States",39.7392,-104.9903
4,ACL,2026,"San Diego, California, United States",32.7157,-117.1611
5,EMNLP,2026,"Budapest, Hungary",47.4979,19.0402
6,AAAI,2026,Singapore EXPO,1.3521,103.8198
7,ECCV,2026,"Malmö, Sweden",55.6050,13.0038
8,IJCAI,2026,"Bremen, Germany",53.0793,8.8017
9,SIGKDD,2026,"Jeju, Korea",33.4996,126.5312


In [4]:
def image_query_value(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def stock_image_url(row: pd.Series) -> str:
    city = image_query_value(row.get("city"))
    country = image_query_value(row.get("country"))
    place = image_query_value(row.get("place"))
    primary = city or place.split(",")[0].strip() or country or "conference city"
    location = ",".join(part for part in [primary, country] if part)
    query = quote(f"{location}, skyline, landmark", safe=",-")
    lock = sum(ord(char) for char in str(row.get("id", primary))) % 10000
    return f"https://loremflickr.com/640/420/{query}?lock={lock}"


def image_file_name(row: pd.Series) -> str:
    safe_id = re.sub(r"[^a-z0-9_-]+", "-", str(row.get("id", "conference")).lower()).strip("-")
    return f"{safe_id}.jpg"


def download_location_image(row: pd.Series) -> str:
    output_path = IMAGE_DIR / image_file_name(row)
    if output_path.exists() and output_path.stat().st_size > 1024:
        return output_path.relative_to(ROOT).as_posix()

    url = row["image_url"]
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "AIConferenceGlobe/1.0 local dataset builder"},
        )
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        image.thumbnail((640, 420))
        canvas = Image.new("RGB", (640, 420), (220, 238, 252))
        left = (640 - image.width) // 2
        top = (420 - image.height) // 2
        canvas.paste(image, (left, top))
        canvas.save(output_path, "JPEG", quality=84, optimize=True)
        return output_path.relative_to(ROOT).as_posix()
    except Exception as exc:
        print(f"Image download failed for {row.get('id')}: {exc}")
        return ""


COLUMNS = [
    "id", "title", "full_name", "year", "subfield", "rank", "importance", "link", "source",
    "deadline", "abstract_deadline", "deadline_timezone",
    "event_start", "event_end", "date_text",
    "place", "city", "country", "latitude", "longitude", "image_url", "image_path",
    "deadline_status", "event_status", "notes",
]

df["image_url"] = df.apply(stock_image_url, axis=1)
df["image_path"] = df.apply(download_location_image, axis=1)
output = df[COLUMNS].copy()
output["importance"] = output["importance"].round(1)
output = output.sort_values(["year", "event_start", "importance"], ascending=[True, True, False])
output_path = DATA_DIR / "conferences.csv"
output.to_csv(output_path, index=False)

print(f"Wrote {len(output)} rows to {output_path}")
print(f"Mapped rows: {output['latitude'].ne('').sum()}")
output.head(20)

Wrote 35 rows to /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/AIConferences/data/conferences.csv
Mapped rows: 35


,id,title,full_name,year,subfield,rank,importance,link,source,deadline,...,date_text,place,city,country,latitude,longitude,image_url,deadline_status,event_status,notes
6,aaai26,AAAI,AAAI Conference on Artificial Intelligence,2026,AI,A,8.2,https://aaai.org/conference/aaai/aaai-26/,CCF Deadlines:conference/AI/aaai.yml,2025-08-01 23:59:59,...,"January 20 - 27, 2026",Singapore EXPO,Singapore,Singapore,1.352100,103.819800,"https://loremflickr.com/640/420/Singapore,Sing...",known,known,
19,wsdm26,WSDM,International Conference on Web Search and Dat...,2026,DM,B,6.8,https://wsdm-conference.org/2026/,CCF Deadlines:conference/DB/wsdm.yml,2025-08-14 23:59:59,...,"February 22 - 26, 2026","Boise, Idaho, USA",Boise,United States,43.615000,-116.202300,"https://loremflickr.com/640/420/Boise,United%2...",known,known,
22,wacv26,WACV,IEEE/CVF Winter Conference on Applications of ...,2026,CV,N,6.5,https://wacv.thecvf.com/,CCF Deadlines:conference/AI/wacv.yml,2025-09-19 23:59:59,...,"Mar 6 - 10, 2026","Tucson, Arizona, USA",Tucson,United States,32.222600,-110.974700,"https://loremflickr.com/640/420/Tucson,United%...",known,known,Second round
23,eacl26,EACL,The Annual Conference of the European Chapter ...,2026,NLP,N,6.4,https://2026.eacl.org/,CCF Deadlines:conference/AI/eacl.yml,2025-10-06 23:59:59,...,"March 24-29, 2026","Rabat, Morocco",Morocco,Morocco,31.791700,-7.092600,"https://loremflickr.com/640/420/Morocco,Morocc...",known,known,
30,ecir26,ECIR,European Conference on Information Retrieval,2026,DB,C,4.0,https://ecir2026.eu/,CCF Deadlines:conference/DB/ecir.yml,2025-10-02 23:59:59,...,"March 30 - April 01, 2026","Delft, The Netherlands",Delft,Nederland,51.999457,4.362724,"https://loremflickr.com/640/420/Delft,Nederlan...",known,known,
2,iclr26,ICLR,International Conference on Learning Represent...,2026,AI,A,9.4,https://iclr.cc/Conferences/2026,CCF Deadlines:conference/AI/iclr.yml,2025-09-24 23:59:59,...,"May 01-05, 2026",Brazil,Rio de Janeiro,Brazil,-22.906800,-43.172900,https://loremflickr.com/640/420/Rio%20de%20Jan...,known,known,
13,aistats26,AISTATS,International Conference on Artificial Intelli...,2026,AI,C,7.4,https://virtual.aistats.org/Conferences/2026,CCF Deadlines:conference/AI/aistats.yml,2025-10-02 23:59:59,...,"May 2-5, 2026",Morocco,Morocco,Morocco,31.791700,-7.092600,"https://loremflickr.com/640/420/Morocco,Morocc...",known,known,
11,icde26,ICDE,IEEE International Conference on Data Engineering,2026,DB,A,7.6,https://icde2026.github.io/,CCF Deadlines:conference/DB/icde.yml,2025-10-27 17:00:00,...,"May 4-8, 2026","Montréal, Canada",Montréal,Canada,45.501900,-73.567400,"https://loremflickr.com/640/420/Montr%C3%A9al,...",known,known,second round
25,aamas26,AAMAS,International Conference on Autonomous Agents ...,2026,AI,B,6.1,https://cyprusconferences.org/aamas2026,CCF Deadlines:conference/AI/aamas.yml,2025-10-08 23:59:59,...,"May 27-29, 2026","Paphos, Cyprus",Paphos,Cyprus,34.772000,32.429700,"https://loremflickr.com/640/420/Paphos,Cyprus,...",known,known,
10,icra26,ICRA,IEEE International Conference on Robotics and ...,2026,RO,B,7.8,https://2026.ieee-icra.org/,CCF Deadlines:conference/AI/icra.yml,2025-09-15 23:59:00,...,"Jun 1-5, 2026","Vienna, Austria",Vienna,Austria,48.208200,16.373800,"https://loremflickr.com/640/420/Vienna,Austria...",known,known,
